# Gene2Wire simulation: group × target blocks — OnDemand 0909

Each repetition generates independent data for each sharing strength. Two balanced artificial groups allow known low-rank structure to be tested under partial target panels.

All existing input gene features are retained. A hidden group × target block contains
both positive and negative assay entries; none becomes a training negative.
The selected targets retain measured training support in other groups, and unselected
eligible targets remain shared. The same mask applies to every cell in a group.

Cells are split **within each group** into training, validation and outer test sets.
Only hidden entries on held-out test cells are scored. Their original references stay
outside preprocessing, calibration, fitting and selection. The full-panel control uses
the same test cells, features, paired cell IDs and tuning rules, but restores native
training/validation availability; it is fitted once and scored on each masked test subset.

This is a separate experiment from the primary 0908/0909 runs. It tests missing target
blocks for new cells within represented groups; it does not test new targets or unseen
animals. Artificial repetitions are not biological replication. No advantage is assumed.

Run top to bottom in an OnDemand Python 3.10+ kernel. Raw data and checkpoints persist;
exports remain under `/home/yueyue/gene2wire/paper_figure_exports`. Figures display here
and save as PDF only. Every code cell has a table-of-contents section.
To redraw completed results, set `RESULTS_ONLY=True` and supply exact saved run directories.
Progress reports once per minute; the worker cell reports active and available slots.


## Run configuration and masking strength


In [ ]:
from pathlib import Path
import os

N_OUTER_FOLDS = 3
USE_LOCATION = False
USE_TARGET_FEATURES = False
N_JOBS = 32
PARALLEL_UNIT = 'scenario'
N_REPETITIONS = 5
STRATEGY = 'full_joint'
CANDIDATE_BUDGET = 32
SEED = 20260909
PAIRED_FRACTION = 0.20

# Fraction of eligible TARGETS receiving a partial group panel, not positive loss.
BLOCK_FRACTIONS = (0.20, 0.40, 0.60, 0.80)
GROUP_MODE = 'artificial'
N_ARTIFICIAL_GROUPS = 2
VALIDATION_FRACTION = 0.20  # Fraction of each group's development cells.
INCLUDE_FULL_PANEL_CONTROL = True

# Isolate structural missingness by default. Optional: (0., .2, .4, .6, .8).
# Nonzero rates use SCAR with matched draws across masked/full training panels.
# Projection-TAGs always retains its natural observed assay and ignores this grid.
POSITIVE_LOSS_RATES = (0.0,)
RUN_INFORMATION_CONTROLS = True
RUN_RANDOM_FOREST = True
RUN_QIAO = True
SHOW_PROGRESS = True
PROGRESS_LEVEL = 'summary'
PROGRESS_INTERVAL_SECONDS = 60.0
SHOW_FULL_DIAGNOSTICS = True

BASE_DIR = Path('/home/yueyue/gene2wire').expanduser()
RAW_DATA_DIR = BASE_DIR / 'raw_data'
CHECKPOINT_DIR = BASE_DIR / 'checkpoints' / 'block_masking_v1'
EXPORT_DIR = BASE_DIR / 'paper_figure_exports'
FIGURE_DIR = BASE_DIR / 'figures' / '0909'
CODE_CACHE_DIR = BASE_DIR / 'code'
RESULTS_ONLY = False
EXISTING_EXPORT_DIRS = {'simulation': None}  # Exact completed run directories.
CORE_COMMIT = '894fab2cee3dcac3e19efcaa9718cf88fd6768dd'
EXPECTED_SOURCE_HASH = '064cf452036aa1562490ee34bd9cdc0f9228499792f11a203e918a377fa240c5'
REPO_URL = 'https://github.com/Yue-stat/Gene2Wire.git'
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
                 'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[variable] = '1'

REQUIRED_MODULES = ['numpy', 'scipy', 'pandas', 'sklearn', 'joblib', 'threadpoolctl', 'matplotlib', 'yaml', 'IPython']
EXPECTED_EXPORT_LABELS = ('simulation',)


## Load and verify the frozen shared core


In [ ]:
import hashlib
import importlib.util
import re
import shutil
import subprocess
import sys
import tempfile

if sys.version_info < (3, 10):
    raise RuntimeError('Select an OnDemand Python 3.10 or newer kernel.')
if not re.fullmatch(r'[0-9a-f]{40}', CORE_COMMIT):
    raise RuntimeError('This notebook needs its released 40-character CORE_COMMIT pin.')
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_SOURCE_HASH):
    raise RuntimeError('This notebook needs its released source checksum.')

def notebook_source_hash(package_root):
    # Same byte-level convention as experiments.protocol.source_hash().
    digest = hashlib.sha256()
    for source_path in sorted(package_root.rglob('*.py')):
        digest.update(source_path.relative_to(package_root).as_posix().encode())
        digest.update(source_path.read_bytes())
    return digest.hexdigest()

def verify_checkout(checkout, require_git_pin=True):
    package_root = checkout / 'src' / 'gene2wire'
    if not package_root.is_dir():
        raise RuntimeError(f'Missing Gene2Wire sources in {checkout}')
    if require_git_pin:
        actual_commit = subprocess.check_output(
            ['git', '-C', str(checkout), 'rev-parse', 'HEAD'], text=True).strip()
        if actual_commit != CORE_COMMIT:
            raise RuntimeError(f'Cached code has commit {actual_commit}, expected {CORE_COMMIT}.')
    if notebook_source_hash(package_root) != EXPECTED_SOURCE_HASH:
        raise RuntimeError(f'Source checksum mismatch in {checkout}; use the released code.')
    return checkout

# Running from the exact local repository is supported without any network call.
CORE_CHECKOUT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    package_root = candidate / 'src' / 'gene2wire'
    if package_root.is_dir() and notebook_source_hash(package_root) == EXPECTED_SOURCE_HASH:
        CORE_CHECKOUT = verify_checkout(candidate, require_git_pin=False)
        break

if CORE_CHECKOUT is None:
    CODE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cached_checkout = CODE_CACHE_DIR / CORE_COMMIT
    if not cached_checkout.exists():
        stage = Path(tempfile.mkdtemp(prefix='.gene2wire-download-', dir=CODE_CACHE_DIR))
        try:
            for arguments in (
                ['git', 'init', '--quiet', str(stage)],
                ['git', '-C', str(stage), 'remote', 'add', 'origin', REPO_URL],
                ['git', '-C', str(stage), 'fetch', '--quiet', '--depth', '1', 'origin', CORE_COMMIT],
                ['git', '-C', str(stage), 'checkout', '--quiet', '--detach', 'FETCH_HEAD'],
            ):
                subprocess.run(arguments, check=True)
            verify_checkout(stage)
            try:
                stage.rename(cached_checkout)
            except OSError:
                # Another notebook may have completed this same immutable cache.
                if not cached_checkout.exists():
                    raise
                verify_checkout(cached_checkout)
        finally:
            if stage.exists():
                shutil.rmtree(stage)
    CORE_CHECKOUT = verify_checkout(cached_checkout)

existing = sys.modules.get('gene2wire')
if existing is not None:
    same_path = Path(existing.__file__).resolve().parent == (CORE_CHECKOUT / 'src' / 'gene2wire').resolve()
    same_source = getattr(existing, '_notebook_source_hash_0908', None) == EXPECTED_SOURCE_HASH
    if not (same_path and same_source):
        raise RuntimeError('A different or unverified gene2wire is already imported. Restart the kernel, then Run All.')

missing = [name for name in REQUIRED_MODULES if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Use an OnDemand Python kernel containing these dependencies: '
                       + ', '.join(missing) + '. See the repository environment instructions.')
sys.path.insert(0, str(CORE_CHECKOUT / 'src'))
import gene2wire
from gene2wire.experiments.protocol import Settings, source_hash
if source_hash() != EXPECTED_SOURCE_HASH:
    raise RuntimeError('Imported code does not match the released source checksum.')
gene2wire._notebook_source_hash_0908 = EXPECTED_SOURCE_HASH
for directory in (RAW_DATA_DIR, CHECKPOINT_DIR, EXPORT_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print({'core_commit': CORE_COMMIT, 'source_hash': source_hash(),
       'imported_from': gene2wire.__file__, 'raw_cache': str(RAW_DATA_DIR),
       'checkpoints': str(CHECKPOINT_DIR), 'exports': str(EXPORT_DIR)})


## Shared model and block settings

All datasets use the same model implementations, candidate budgets and exact Joint
endpoints. Only group source, native availability and feature adapters differ.
Paired references authorize visible entries of the same selected cells, never blocked
entries. Nonzero optional positive loss uses simple SCAR, separately from structural masks.
Qiao uses target identity when target features are disabled; enabling the switch requires
the adapter's declared outcome-independent target descriptors.


In [ ]:
from dataclasses import asdict
import numpy as np
import pandas as pd
from IPython.display import display
from gene2wire.experiments.block_design import BlockMaskConfig
from gene2wire.experiments.block_experiment import (
    run_block_experiment, run_block_simulation_experiments, preview_block_experiment,
)
from gene2wire.experiments.block_plotting import plot_block_results
from gene2wire.experiments.reporting import (
    configure_full_display, configure_compact_display, load_existing_exports,
)
from gene2wire.tuning import full_joint_candidates

if SHOW_FULL_DIAGNOSTICS:
    configure_full_display()
else:
    configure_compact_display()
settings = Settings(
    n_outer_folds=N_OUTER_FOLDS, use_location=USE_LOCATION,
    use_target_features=USE_TARGET_FEATURES, n_jobs=N_JOBS,
    parallel_unit=PARALLEL_UNIT, n_repetitions=N_REPETITIONS,
    strategy=STRATEGY, candidate_budget=CANDIDATE_BUDGET, seed=SEED,
    paired_fraction=PAIRED_FRACTION, calibration_fractions=(PAIRED_FRACTION,),
    loss_rates=POSITIVE_LOSS_RATES,
    run_information_controls=RUN_INFORMATION_CONTROLS,
    run_random_forest=RUN_RANDOM_FOREST, run_qiao=RUN_QIAO,
    run_mechanism_controls=False, run_calibration_controls=False,
)
block_config = BlockMaskConfig(
    fractions=BLOCK_FRACTIONS, group_mode=GROUP_MODE,
    n_artificial_groups=N_ARTIFICIAL_GROUPS,
    validation_fraction=VALIDATION_FRACTION,
    include_full_panel_control=INCLUDE_FULL_PANEL_CONTROL,
)
display(pd.DataFrame([asdict(settings)]).T.rename(columns={0: 'shared settings'}))
display(pd.DataFrame([asdict(block_config)]).T.rename(columns={0: 'block design'}))
if RESULTS_ONLY:
    all_artifacts = load_existing_exports(
        EXISTING_EXPORT_DIRS, expected_labels=EXPECTED_EXPORT_LABELS)
    print('RESULTS_ONLY: using saved manifests; raw loading and fitting are skipped.')


## Live worker usage


In [ ]:
from gene2wire.experiments.workers import NotebookWorkerStatus

worker_status = None
if RESULTS_ONLY:
    print('RESULTS_ONLY: no training workers are launched.')
else:
    worker_status = NotebookWorkerStatus(requested_workers=N_JOBS)


## Preview design and tuning candidates


In [ ]:
def preflight_blocks(dataset):
    dataset.validate()
    preview = preview_block_experiment(dataset, settings, block_config, repetition=0)
    groups, folds, design = preview['groups'], preview['folds'], preview['design']
    features = dataset.feature_builder(
        folds[0].train_rows, settings.use_location, settings.use_target_features)
    display(pd.DataFrame([{
        'dataset': dataset.name, 'cells': len(dataset.cell_ids),
        'targets': len(dataset.target_ids), 'groups': len(np.unique(groups)),
        'feature_columns': features.X.shape[1],
        'feature_blocks': dict(features.feature_blocks),
        'target_feature_columns': 0 if features.Y_target is None else features.Y_target.shape[1],
        'native_measured_pairs': int(dataset.measured.sum()),
        'natural_paired_assay': dataset.natural_observed is not None,
    }]))
    display(pd.Series(groups).value_counts().rename_axis('group').to_frame('cells'))
    print('Repetition 0 design; actual fractions account for rounded target counts and native coverage:')
    display(design.summary)
    display(pd.DataFrame([
        {'fold': f.outer_fold, 'inner_train': len(f.train_rows),
         'validation': len(f.validation_rows), 'test': len(f.test_rows),
         'split': f.metadata['split_type']} for f in folds]))
    if SHOW_FULL_DIAGNOSTICS:
        display(design.assignment)
    tuning = settings.tuning_config(features.X.shape[1], len(dataset.target_ids))
    candidates = []
    for model in settings.models():
        candidates.extend({'model': model.name, 'candidate': index, **asdict(candidate)}
                          for index, candidate in enumerate(full_joint_candidates(model, tuning), 1))
    candidate_table = pd.DataFrame(candidates)
    if SHOW_FULL_DIAGNOSTICS:
        display(candidate_table)
        display(pd.DataFrame([asdict(settings.fit_config())]))
    else:
        display(candidate_table.groupby('model').size().to_frame('candidates'))
    print('No input genes are masked. Each group appears in training, validation and test.')
    print('Blocked entries are excluded from calibration, model fitting and validation scoring.')


## Simulation truth and input settings

The location switch controls both generated projection signal and fitted features.
Sharing strengths 0, 0.5 and 1 are all retained. The paired subset defaults to 20%.
Generated data are persisted and reused; the preview below is only a small design audit.


In [ ]:
if not RESULTS_ONLY:
    from gene2wire.experiments.datasets.simulation import generate_simulation
    SHARING_STRENGTHS = (0.0, 0.5, 1.0)
    SIMULATION_OPTIONS = {
        'n_cells': 400, 'n_targets': 36, 'n_gene_features': 16,
        'n_location_features': 4, 'n_slices': 20, 'true_rank': 2,
        'n_target_features': 4, 'signal_sd': 1.35,
        'target_feature_noise': 0.85,
        'target_prevalence_range': (0.10, 0.30),
        'truth_uses_location': USE_LOCATION,
    }
    representative = generate_simulation(
        repetition=0, sharing_strength=SHARING_STRENGTHS[0], seed=SEED,
        **SIMULATION_OPTIONS)
    preflight_blocks(representative)


## Run all block settings and the full-panel control

This is the expensive step. A model unit includes its tuning and final refit.
Compatible completed units and partial fit caches resume automatically. The full-panel
control is not refitted for every evaluation fraction. At zero additional thinning,
ordinary and PU objectives can coincide; overlapping curves are expected.


In [ ]:
if not RESULTS_ONLY:
    artifacts = run_block_simulation_experiments(
        settings=settings, block_config=block_config,
        raw_cache_dir=RAW_DATA_DIR / 'simulation_block_masking',
        checkpoint_dir=CHECKPOINT_DIR, export_dir=EXPORT_DIR,
        sharing_strengths=SHARING_STRENGTHS,
        simulation_options=SIMULATION_OPTIONS,
        progress=SHOW_PROGRESS, progress_level=PROGRESS_LEVEL,
        progress_interval=PROGRESS_INTERVAL_SECONDS, worker_status=worker_status)
    all_artifacts = {'simulation': artifacts}


## Curves on structurally hidden test entries

Panels show AUPRC, log loss and Brier score against the fraction of eligible targets
assigned a partial panel. The full-panel row uses the same hidden test subsets, so its
curve can change as evaluation targets change even though its fitted model is reused.
All scores use raw predictions; a missing assay is not an observed negative and receives
no non-detection posterior. Thus Hidden Recall@H is not used in this experiment.
Additional plots show all benchmarks, methods without direct reference supervision,
and the three same-budget reference-plus-observation methods. Curves are descriptive
averages over folds and repetitions, not confidence intervals across animals.


In [ ]:
figure_paths = {}
for label, artifacts in all_artifacts.items():
    figure_paths[label] = plot_block_results(
        artifacts, output_dir=FIGURE_DIR, show=True, include_benchmarks=True)
display(figure_paths)


## Export inventory, mask audits, metrics and selected hyperparameters


In [ ]:
for label, artifacts in all_artifacts.items():
    print(f'{label}: {artifacts.export_dir}')
    display(pd.DataFrame([{'table': name, 'rows': len(frame), 'columns': len(frame.columns)}
                          for name, frame in artifacts.tables.items()]))
    names = (list(artifacts.tables) if SHOW_FULL_DIAGNOSTICS else
             ['block_summary', 'aggregate', 'selected', 'failures'])
    for name in names:
        frame = artifacts.tables.get(name)
        if frame is not None:
            print(f'{name}: {len(frame)} rows')
            display(frame)
    if SHOW_FULL_DIAGNOSTICS:
        import json
        print(json.dumps(artifacts.manifest, indent=2, default=str))
print('PDF figures:', FIGURE_DIR)
print('Persistent checkpoints:', CHECKPOINT_DIR)
print('Reusable raw data:', RAW_DATA_DIR)
